In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
CATALOG = "olist_dw_project"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

#customers

## 1. exploration

In [0]:
df_cust = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
print(df_cust.count())
display(df_cust.limit(15))

In [0]:
df_cust.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_cust.columns]).show()

In [0]:
print(df_cust.select("customer_id").distinct().count())
print(df_cust.select("customer_unique_id").distinct().count())

In [0]:
display(df_cust.select("customer_state").distinct())

## 2.transformation

In [0]:
display(df_cust.limit(10))

In [0]:
brazil_state_map = {
    "SP": "Sao Paulo", "SC": "Santa Catarina", "MG": "Minas Gerais",
    "PR": "Parana", "RJ": "Rio de Janeiro", "RS": "Rio Grande do Sul",
    "PA": "Para", "GO": "Goias", "ES": "Espirito Santo",
    "BA": "Bahia", "MA": "Maranhao", "MS": "Mato Grosso do Sul",
    "CE": "Ceara", "DF": "Distrito Federal", "RN": "Rio Grande do Norte",
    "PE": "Pernambuco", "MT": "Mato Grosso", "AM": "Amazonas",
    "AP": "Amapa", "AL": "Alagoas", "RO": "Rondonia",
    "PB": "Paraiba", "TO": "Tocantins", "PI": "Piaui",
    "AC": "Acre", "SE": "Sergipe", "RR": "Roraima",
}
mapping_expr = F.create_map([F.lit(x) for pair in brazil_state_map.items() for x in pair])

In [0]:
df_cust_silver = (
    df_cust
    .select(
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
        F.trim(F.col("customer_zip_code_prefix")).alias("customer_zip_code"),
        F.initcap(
            F.regexp_replace(F.trim(F.col("customer_city")), r"\s+", " ")
        ).alias("customer_city"),
        mapping_expr[F.upper(F.trim(F.col("customer_state")))].alias("customer_state"),
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

In [0]:
display(df_cust_silver.limit(10))

## 3. Check

In [0]:
print(f"Bronze row count:  {df_cust.count():,}")
print(f"Silver row count:  {df_cust_silver.count():,}")

In [0]:
df_cust_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_cust_silver.columns]).show()

In [0]:
dup_check = (
    df_cust_silver.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate customer_id count: {dup_check}")

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.customers"
(
    df_cust_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

#sellers


## 1. Exploration

In [0]:
df_sellers = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.sellers")
print(df_sellers.count())
display(df_sellers.limit(15))

In [0]:
df_sellers.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_sellers.columns]).show()

In [0]:
print(df_sellers.select("seller_id").distinct().count())

## 2.transformation

In [0]:
display(df_sellers.limit(10))

In [0]:
df_sellers_silver = (
    df_sellers
    .select(
        F.trim(F.col("seller_id")).alias("seller_id"),
        F.trim(F.col("seller_zip_code_prefix")).alias("seller_zip_code"),
        F.initcap(
            F.regexp_replace(F.trim(F.col("seller_city")), r"\s+", " ")
        ).alias("seller_city"),
        mapping_expr[F.upper(F.trim(F.col("seller_state")))].alias("seller_state"),
    )
    .filter(F.col("seller_id").isNotNull())
    .dropDuplicates(["seller_id"])  
)

In [0]:
display(df_sellers_silver.limit(10))

## 3. check

In [0]:
print(f"Bronze row count:  {df_sellers.count():,}")
print(f"Silver row count:  {df_sellers_silver.count():,}")

In [0]:
df_sellers_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_sellers_silver.columns]).show()

In [0]:
dup_check = (
    df_sellers_silver.groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate seller_id count: {dup_check}")

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.sellers"
(
    df_sellers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

#order items

## 1. exploration

In [0]:
df_item = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_items")
print(df_item.count())
display(df_item.limit(15))

In [0]:
df_item.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_item.columns]).show()

In [0]:
print(df_item.select("order_id").distinct().count())
print(df_item.select("order_id", "order_item_id").distinct().count())

## 2. transformation

In [0]:
display(df_item.limit(5))

In [0]:
df_item_silver = (
    df_item
    .select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.col("order_item_id").cast(IntegerType()).alias("order_item_id"),
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.col("seller_id")).alias("seller_id"),
        F.to_timestamp("shipping_limit_date").alias("shipping_limit_ts"),
        F.col("price").cast(DecimalType(10, 2)).alias("price"),
        F.col("freight_value").cast(DecimalType(10, 2)).alias("freight_value"),
    )
    .withColumn("total", F.col("price") + F.col("freight_value"))
    .filter(
        F.col("order_id").isNotNull() 
        & F.col("order_item_id").isNotNull()
        & F.col("product_id").isNotNull()
    )
    .dropDuplicates(["order_id", "order_item_id"])
    .filter((F.col("price") >= 0) & (F.col("freight_value") >= 0))
)

In [0]:
display(df_item_silver.limit(5))

## 3.check

In [0]:
print(f"Bronze row count:  {df_item.count():,}")
print(f"Silver row count:  {df_item_silver.count():,}")
df_item_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_item_silver.columns]).show()
dup_check = (
    df_item_silver.groupBy("order_id","order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate pk count: {dup_check}")

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.order_items"
(
    df_item_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

#order payments

## 1. explorarion

In [0]:
df_pay = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.order_payments")
print(df_pay.count())
display(df_pay.limit(15))

In [0]:
df_pay.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_pay.columns]).show()

In [0]:
print(df_pay.select("order_id").distinct().count())
print(df_pay.select("order_id","payment_sequential").distinct().count())

In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS actual_payment_count,
    MAX(payment_sequential) AS max_sequential,
    MIN(payment_sequential) AS min_sequential
FROM olist_dw_project.silver.order_payments
GROUP BY order_id
HAVING COUNT(*) != MAX(payment_sequential)
ORDER BY (MAX(payment_sequential) - COUNT(*)) DESC;

## 2.transformation

In [0]:
display(df_pay.limit(5))

In [0]:
df_pay_silver = (
    df_pay.select
    (F.trim(F.col("order_id")).alias("order_id"),
     F.col("payment_sequential").cast(IntegerType()).alias("payment_sequential"),
     F.trim(F.regexp_replace(F.lower(F.col("payment_type")), "_", " ")).alias("payment_type"),
     F.col("payment_installments").cast(IntegerType()).alias("payment_installments"),
     F.col("payment_value").cast(DecimalType(10, 2)).alias("payment_value")
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("payment_sequential").isNotNull()
    )
    .dropDuplicates(["order_id", "payment_sequential"])
    .filter(F.col("payment_value") >= 0)
    .withColumn(
        "payment_sequential",
        F.row_number().over(window_spec)
    )
    .withColumn(
        "payment_installments",
        F.when(F.col("payment_installments") < 1, 1)
         .otherwise(F.col("payment_installments"))
    )
)


In [0]:
display(df_pay_silver.limit(5))

## check

In [0]:
print(f"Bronze row count:  {df_pay.count():,}")
print(f"Silver row count:  {df_pay_silver.count():,}")
df_pay_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_pay_silver.columns]).show()
dup_check = (
    df_pay_silver.groupBy("order_id","payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate pk count: {dup_check}")

In [0]:
df_pay_silver.select("payment_type").distinct().sort("payment_type").show()

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.order_payments"
(
    df_pay_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

#orders

## 1. exploration

In [0]:
df_order = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders")                      
print(df_order.count())
display(df_order.limit(15))

In [0]:
df_order.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_order.columns]).show()

In [0]:
display(df_order.select("order_status").groupBy("order_status").count())

1. order_purchase_ts: ลูกค้ากดสั่งซื้อ (ต้องมีเสมอ)
2. order_approved_ts: ระบบอนุมัติการจ่ายเงิน
3. order_delivered_carrier_ts ← ส่งมอบให้ขนส่ง 
4. order_delivered_customer_ts ← ลูกค้าได้รับของจริง
5. order_estimated_delivery_ts ← วันที่คาดว่าจะส่งถึง

6. status
- created: เพิ่งสั่งซื้อแต่ยังไม่approve
- approved, processing, invoiced, unavailable: ยังไม่ส่ง
- shipped: ส่งแล้วยังไม่ถึง
- delivered: ส่งถึงเรียบร้อย
- cancled: ไม่ approved/ ยกเลิกก่อนส่ง/ ยกเลิกหลังส่ง

In [0]:
display(df_order.filter(F.col("order_approved_at").isNull()).groupBy("order_status").count())
display(df_order.filter(F.col("order_approved_at").isNull() & F.col("order_delivered_carrier_date").isNotNull()))
display(df_order.filter(F.col("order_approved_at").isNull() & (F.col("order_status")=="canceled")))
display(df_order.filter(F.col("order_approved_at").isNull() & (F.col("order_status")=="created")))

In [0]:
print(df_order.filter(F.col("order_status")=="canceled").count())
print(df_order.filter(F.col("order_status")=="created").count())

In [0]:
display(df_order.filter(
    F.col("order_approved_at").isNotNull() & F.col("order_delivered_carrier_date").isNull())
        .groupby("order_status").count())

In [0]:
display(df_order.filter(
    F.col("order_approved_at").isNotNull() & F.col("order_delivered_carrier_date").isNull()
    & F.col("order_status").isin(["delivered"])
))

In [0]:
display(
    df_order.filter(
        (F.col("order_delivered_customer_date").isNull()) & 
        (F.col("order_approved_at").isNotNull()) & 
        (F.col("order_delivered_carrier_date").isNotNull())
    )
    .groupBy("order_status")
    .count()
)

In [0]:
display(
    df_order.filter(
        (F.col("order_delivered_customer_date").isNull()) & 
        (F.col("order_approved_at").isNotNull()) & 
        (F.col("order_delivered_carrier_date").isNotNull()) &
        (F.col("order_status")=="delivered"))
    )

In [0]:
display(
    df_order.filter(
        (F.col("order_delivered_customer_date").isNotNull()) & 
        (F.col("order_approved_at").isNotNull()) & 
        (F.col("order_delivered_carrier_date").isNotNull()) &
        (F.col("order_status")=="canceled"))
)

In [0]:
display(
    df_order.filter(
        (F.col("order_status") == "delivered") & 
        (
            F.col("order_delivered_customer_date").isNull() | 
            F.col("order_approved_at").isNull() | 
            F.col("order_delivered_carrier_date").isNull()
        )
    )
)

In [0]:
print(df_order.select("order_id").distinct().count())
print(df_order.select("customer_id").distinct().count())

## 2.transformation

In [0]:
display(df_order.limit(5))

In [0]:
df_order_silver = (
    df_order
    .select(
        F.trim(F.col("order_id")).alias("order_id"),
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.lower(F.col("order_status"))).alias("order_status"),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.to_timestamp("order_approved_at").alias("order_approved_ts"),
        F.to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_ts"),
        F.to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_ts"),
        F.to_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_ts"),
    )
    .filter(
        F.col("order_id").isNotNull() 
        & F.col("customer_id").isNotNull()
        & F.col("order_purchase_ts").isNotNull()
    )
    .dropDuplicates(["order_id"])
)

In [0]:
display(df_order_silver.limit(20))

In [0]:
anomalies = df_order_silver.filter(
    (F.col("order_approved_ts") < F.col("order_purchase_ts"))
    | (F.col("order_delivered_carrier_ts") < F.col("order_approved_ts"))
    | (F.col("order_delivered_customer_ts") < F.col("order_delivered_carrier_ts"))
    | (F.col("order_estimated_delivery_ts") < F.col("order_purchase_ts"))
)

print(f"Rows with illogical timestamp order: {anomalies.count()}")
display(anomalies.limit(10))

In [0]:
df_order_silver.groupBy("order_status").agg(
    F.count("*").alias("total"),
    F.sum(F.col("order_approved_ts").isNull().cast("int")).alias("null_approved"),
    F.sum(F.col("order_delivered_carrier_ts").isNull().cast("int")).alias("null_carrier"),
    F.sum(F.col("order_delivered_customer_ts").isNull().cast("int")).alias("null_delivered"),
).orderBy("order_status").show()

In [0]:
display(df_order_silver.filter(
    F.col("order_delivered_customer_ts").isNotNull() 
    & F.col("order_delivered_carrier_ts").isNull()
))

display(df_order_silver.filter(
    F.col("order_delivered_carrier_ts").isNotNull() 
    & F.col("order_approved_ts").isNull()
))

In [0]:
approved_expected_statuses = ["approved", "processing", "invoiced", "shipped", "delivered"]
carrier_expected_statuses = ["shipped", "delivered"]

display(df_order_silver.filter(
    (F.col("order_status").isin(approved_expected_statuses) & F.col("order_approved_ts").isNull()) | 
    (F.col("order_status").isin(carrier_expected_statuses) & F.col("order_delivered_carrier_ts").isNull()) |
    ((F.col("order_status") == "delivered") & F.col("order_delivered_customer_ts").isNull())
))

In [0]:
import pyspark.sql.functions as F

df_order_silver.filter(
    (
        F.unix_timestamp("order_delivered_customer_ts")
        - F.unix_timestamp("order_purchase_ts")
    )
    > (210 * 86400)
).show()

In [0]:
df_order_silver.groupBy("customer_id", "order_purchase_ts", "order_delivered_customer_ts") \
  .count() \
  .filter(F.col("count") > 1) \
  .show(20, truncate=False)

In [0]:
display(df_order_silver.filter((F.col("order_status") != "delivered") & (F.col("order_delivered_customer_ts").isNotNull())))

In [0]:
display(df_order_silver.groupBy("order_approved_ts").count() \
    .filter(F.col("count") > 5) \
    .orderBy(F.desc("count")))

## timestamp transformation

### 1. Worng Sequence 

**Issues found:**
- `order_approved_ts`: duplicate timestamps (up to 9x/sec) with no business explanation → batch job artifact
- `order_delivered_carrier_ts`: duplicates too (up to 47x), but explained by real batch shipment pickups → more trustworthy
- `order_delivered_customer_ts`: confirmed by actual delivery → most trustworthy
- 7 rows: status=`delivered` but `delivered_customer_ts` is null → conflicting data, dropped
- ~15 rows: isolated missing `approved`/`carrier` → left as null (no data to infer from)

**Fixes (applied in order: purchase → approved → carrier → customer):**
| Check | Fix |
|---|---|
| `approved_ts < purchase_ts` | none needed (0 rows) |
| `carrier_ts < approved_ts` | null `approved_ts` |
| `customer_ts < carrier_ts` | null `carrier_ts` |
| `delivered` + `customer_ts` null | drop row |

**Principle:** fix upstream→downstream

In [0]:
invalid_approved = df_order_silver.filter(
    F.col("order_approved_ts") < F.col("order_purchase_ts")
).count()
print(f"Rows with approved_ts before purchase_ts: {invalid_approved}")

In [0]:
invalid_carrier_approved = df_order_silver.filter(
    F.col("order_delivered_carrier_ts") < F.col("order_approved_ts")
).count()
print(f"Rows with carrier_ts < approved_ts: {invalid_carrier_approved}")

In [0]:
df_order_silver = df_order_silver.withColumn(
    "order_approved_ts",
    F.when(
        F.col("order_delivered_carrier_ts") < F.col("order_approved_ts"),
        F.lit(None)
    ).otherwise(F.col("order_approved_ts"))
)

# recheck
still_invalid = df_order_silver.filter(
    F.col("order_delivered_carrier_ts") < F.col("order_approved_ts")
).count()
print(f"Remaining violations after fix: {still_invalid}") 

In [0]:
invalid_customer_carrier = df_order_silver.filter(
    F.col("order_delivered_customer_ts") < F.col("order_delivered_carrier_ts")
).count()
print(f"Rows with customer_ts < carrier_ts: {invalid_customer_carrier}")

In [0]:
df_order_silver = df_order_silver.withColumn(
    "order_delivered_carrier_ts",
    F.when(
        F.col("order_delivered_customer_ts") < F.col("order_delivered_carrier_ts"),
        F.lit(None)
    ).otherwise(F.col("order_delivered_carrier_ts"))
)

# recheck
still_invalid = df_order_silver.filter(
    F.col("order_delivered_customer_ts") < F.col("order_delivered_carrier_ts")
).count()
print(f"Remaining violations after fix: {still_invalid}") 

In [0]:
invalid_estimated = df_order_silver.filter(
    F.col("order_estimated_delivery_ts") < F.col("order_purchase_ts")
).count()
print(f"Rows with estimated_ts < purchase_ts: {invalid_estimated}")

### 2. Inconsistency status vs ts 

**หลักการ:** `order_status` คือ final state ที่เชื่อถือได้ แต่ timestamp ที่มีอยู่คือบันทึกเหตุการณ์จริง
ที่เคยเกิดขึ้น — status บอกแค่ "จบลงอย่างไร" ไม่ได้ลบล้างว่า "อะไรเคยเกิดขึ้นระหว่างทาง"
เช่น order ที่ถูก `canceled` อาจเคย `delivered` มาก่อน (ยกเลิก/คืนสินค้าหลังได้รับของ) เพราะอันที่มีวันส่งแต่ status ไม่ delivered มีแค่ cancled

**สรุป: ไม่ drop/null timestamp ใดๆ ในหมวดนี้เลย** — เก็บข้อมูลตามที่ Bronze ให้มาทั้งหมด

**สิ่งที่พบ:**
| กรณี | จำนวน | หมายเหตุ |
|---|---|---|
| `delivered` + timestamp ขาดบางส่วน | หลายพันแถว (ส่วนใหญ่จาก sequence fix หมวด 1) | บันทึกไม่ครบ |
| `canceled` + ยังมี `delivered_customer_ts`/`carrier_ts` | 69 | ยกเลิกหลังส่งของ/คืนสินค้า — เป็นไปได้จริง |
| `unavailable` + มี `approved_ts` | มีอยู่จริง | อนุมัติแล้วแต่สินค้าหมดทีหลัง |
| `created` + มี `approved_ts` | 0 | สมเหตุสมผล (ยังไม่ถึงขั้นอนุมัติ) |

### 3. Anomalous Numeric

**หลักการ:** 
- ช่วงเวลาระหว่าง purchase_ts ถึง delivered_customer_ts ผิดปกติ (เช่น 0 วินาทีหรือหลายปี)
- Duplicate timestamp เหมือนกันเป๊ะๆ ทุกคอลัมน์ 

**สรุป:** max 7 เดือน(200วัน) พอรับได้และไม่มี duplicate


In [0]:
df_order_silver.withColumn(
    "delivery_days",
    (F.col("order_delivered_customer_ts").cast("long") - F.col("order_purchase_ts").cast("long")) / 86400
).select("delivery_days").summary("min", "25%", "50%", "75%", "max").show()

In [0]:
df_order_silver.withColumn(
    "delivery_days",
    (F.col("order_delivered_customer_ts").cast("long") - F.col("order_purchase_ts").cast("long")) / 86400
).select("delivery_days").filter(
    (F.col("delivery_days") > 0) & (F.col("delivery_days") >= 60)
).count()

In [0]:
suspicious_duplicates = df_order_silver.filter(
    (F.col("order_purchase_ts") == F.col("order_approved_ts"))
    & (F.col("order_approved_ts") == F.col("order_delivered_carrier_ts"))
    & (F.col("order_delivered_carrier_ts") == F.col("order_delivered_customer_ts"))
).count()
print(f"Rows with all timestamps identical: {suspicious_duplicates}")

In [0]:
df_order_silver.groupBy("order_status").agg(
    F.count("*").alias("total"),
    F.sum(F.col("order_approved_ts").isNull().cast("int")).alias("null_approved"),
    F.sum(F.col("order_delivered_carrier_ts").isNull().cast("int")).alias("null_carrier"),
    F.sum(F.col("order_delivered_customer_ts").isNull().cast("int")).alias("null_delivered"),
).orderBy("order_status").show()

In [0]:
df_order_silver = (
    df_order_silver
    .withColumn(
        "delivery_days", 
        F.datediff("order_delivered_customer_ts", "order_purchase_ts")
    )
    .withColumn(
        "is_delayed", 
        F.col("order_delivered_customer_ts") > F.col("order_estimated_delivery_ts")
    )
)

In [0]:
display(df_order_silver.limit(20))

## 3. check

In [0]:
print(f"Bronze row count:  {df_order.count():,}")
print(f"Silver row count:  {df_order_silver.count():,}")
df_order_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_order_silver.columns]).show()
dup_check = (
    df_order_silver.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate pk count: {dup_check}")

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.orders"
(
    df_order_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

#products

## 1. exploration

In [0]:
df_prod = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.products")
print(df_prod.count())
display(df_prod.limit(15))

In [0]:
df_prod.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_prod.columns]).show()

In [0]:
display(
    df_prod.filter(
        F.col("product_category_name").isNotNull() 
        & F.col("product_name_lenght").isNull()
    ).limit(30)
)

In [0]:
display(df_prod.filter(F.col("product_weight_g").isNull()).limit(30))

In [0]:
empty_products = df_prod.filter(
    F.col("product_category_name").isNull()
    & F.col("product_weight_g").isNull()
    & F.col("product_length_cm").isNull()
    & F.col("product_height_cm").isNull()
    & F.col("product_width_cm").isNull()
)
print(f"Products with only product_id, no other data: {empty_products.count()}")
empty_products.show(truncate=False)

# เช็คว่า product นี้เคยถูกซื้อขายจริงมั้ย (join กับ order_items)
df_order_items_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.order_items")

empty_products_with_orders = empty_products.join(
    df_order_items_silver,
    on="product_id",
    how="inner"
)
print(f"Of those, how many appear in order_items (real transactions): {empty_products_with_orders.count()}")

In [0]:
display(
    df_prod.filter(
        (F.col("product_weight_g").cast("integer") <= 0) |
        (F.col("product_length_cm").cast("integer") <= 0) |
        (F.col("product_height_cm").cast("integer") <= 0)
    )
)

In [0]:
display(df_prod.filter(F.col("product_category_name").isNull()).limit(30))

In [0]:
print(df_prod.select("product_id").distinct().count())

## 2.transformation

In [0]:
display(df_prod.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

df_products_silver = (
    df_prod
    .select(
        F.trim(F.col("product_id")).alias("product_id"),

        # product_category_name เป็น null ใส่ 'n/a' แทน
        F.coalesce(
            F.trim(
                F.regexp_replace(F.lower(F.col("product_category_name")), "_", " ")
            ),
            F.lit("n/a")
        ).alias("product_category_name"),
        
        # product_name_lenght เป็น null ใส่ 0
        F.coalesce(
            F.col("product_name_lenght").cast(IntegerType()),
            F.lit(0)
        ).alias("product_name_length"),
        
        # product_description_lenght เป็น null ใส่ 0
        F.coalesce(
            F.col("product_description_lenght").cast(IntegerType()),
            F.lit(0)
        ).alias("product_description_length"),
        
        # product_photos_qty เป็น null ใส่ 'n/a' แทน 
        F.coalesce(
            F.col("product_photos_qty").cast(StringType()),
            F.lit("n/a")
        ).alias("product_photos_qty"),
        
        # (<= 0 ให้เป็น null)
        F.when(F.col("product_weight_g") > 0, F.col("product_weight_g").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_weight_g"),
        F.when(F.col("product_length_cm") > 0, F.col("product_length_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_length_cm"),
        F.when(F.col("product_height_cm") > 0, F.col("product_height_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_height_cm"),
        F.when(F.col("product_width_cm") > 0, F.col("product_width_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_width_cm"),
    )
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

In [0]:
df_translation = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.product_category")

In [0]:
# distinct category names
products_categories = (df_prod.select(F.trim(F.lower(F.col("product_category_name"))).alias("category")).distinct())
translation_categories = (df_translation.select(F.trim(F.lower(F.col("product_category_name"))).alias("category")).distinct())

# category ใน products ที่ "ไม่มี" อยู่ใน translation table (left anti join)
unmatched = products_categories.join(
    translation_categories,
    on="category",
    how="left_anti"
)

print(f"Categories in products but missing in translation table: {unmatched.count()}")
unmatched.show(50, truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType

df_translation = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.product_category")

df_products_silver = (
    df_prod
    .select(
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.lower(F.col("product_category_name"))).alias("product_category_name"),

        F.coalesce(
            F.col("product_name_lenght").cast(IntegerType()),
            F.lit(0)
        ).alias("product_name_length"),

        F.coalesce(
            F.col("product_description_lenght").cast(IntegerType()),
            F.lit(0)
        ).alias("product_description_length"),

        F.coalesce(
            F.col("product_photos_qty").cast(IntegerType()),
            F.lit(0)
        ).alias("product_photos_qty"),

        F.when(F.col("product_weight_g") > 0, F.col("product_weight_g").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_weight_g"),
        F.when(F.col("product_length_cm") > 0, F.col("product_length_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_length_cm"),
        F.when(F.col("product_height_cm") > 0, F.col("product_height_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_height_cm"),
        F.when(F.col("product_width_cm") > 0, F.col("product_width_cm").cast(IntegerType()))
         .otherwise(F.lit(None)).alias("product_width_cm"),
    )
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
    .join(
        df_translation.select(
            F.trim(F.lower(F.col("product_category_name"))).alias("product_category_name"),
            F.trim(F.col("product_category_name_english")).alias("product_category_name_english")
        ),
        on="product_category_name",
        how="left"
    )
    .withColumn(
        "product_category_name",
        F.coalesce(F.regexp_replace(F.col("product_category_name"), "_", " "), F.lit("n/a"))
    )
    .withColumn(
        "product_category",
        F.coalesce(F.regexp_replace(F.col("product_category_name_english"), "_", " "), F.col("product_category_name"))
    )
    .select(
        "product_id",
        "product_category",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    )
)

In [0]:
display(df_products_silver)

## 3. check

In [0]:
print(f"Bronze row count:  {df_prod.count():,}")
print(f"Silver row count:  {df_products_silver.count():,}")
df_products_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_products_silver.columns]).show()
dup_check = (
    df_products_silver.groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate pk count: {dup_check}")

In [0]:
target_table = f"{CATALOG}.{SILVER_SCHEMA}.products"
(
    df_products_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)